Time Series Project: Matthew Duffy, Sarah Her, Paula Reece, and Yvonne Zhang

Load Data

In [3]:
!pip install arch

In [4]:
import warnings
warnings.filterwarnings("ignore")
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter("ignore", ConvergenceWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from arch import arch_model
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import t
import pandas_datareader.data as web
from sklearn.metrics import mean_squared_error, mean_absolute_error
from google.colab import drive

drive.mount('/content/drive')

df1_daily = pd.read_excel('/content/drive/MyDrive/Time Series Project/Time Series Project Dataset.xlsx', sheet_name='Daily Data')
df1_monthly = pd.read_excel('/content/drive/MyDrive/Time Series Project/Time Series Project Dataset.xlsx', sheet_name='Monthly Data')
df2 = pd.read_csv('/content/drive/MyDrive/Time Series Project/Bitcoin_history_data.csv')

# Rename columns

df1_daily['Date'] = pd.to_datetime(df1_daily['Date'])
df1_monthly["Month"] = pd.to_datetime(df1_monthly["Month"])
df2['Date'] = pd.to_datetime(df2['Date'])

# Set indices
daily   = df1_daily.set_index("Date").sort_index()
monthly = df1_monthly.set_index("Month").sort_index()
btc     = (df2.set_index("Date").sort_index())

# 2. Merge Monthly → Daily
# ============================
# Forward-fill monthly data onto daily index
monthly_as_daily = monthly.reindex(daily.index, method="ffill")
merged = (daily
          .join(btc, how="left")
          .join(monthly_as_daily, how="left"))

merged.head(20)

MessageError: Error: credential propagation was unsuccessful

Baseline Model - ARIMA with no exogenous variable

In [ ]:
split_date = merged.index.max() - pd.DateOffset(months=2)
train = merged.loc[:split_date]
test = merged.loc[split_date+pd.Timedelta(days=1):]


print("Train range:", train.index.min(), "to", train.index.max())
print("Test range:", test.index.min(), "to", test.index.max())


# Define target variable (SPY Price)
y_train = train["SPY Price"]
y_test = test["SPY Price"]

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt

# Use returns (log or pct change) for stationarity check
returns = train["SPY Price"].pct_change().dropna()

# ---- ADF Test ----
adf_result = adfuller(returns)
print("ADF Statistic:", adf_result[0])
print("p-value:", adf_result[1])
for key, value in adf_result[4].items():
    print("Critical Value (%s): %.3f" % (key, value))

if adf_result[1] < 0.05:
    print("✅ Stationary (reject null)")
else:
    print("⚠️ Non-stationary (consider differencing)")

# ---- ACF/PACF Plots ----
fig, axes = plt.subplots(1, 2, figsize=(12,4))
plot_acf(returns, lags=30, ax=axes[0])
plot_pacf(returns, lags=30, ax=axes[1])
plt.show()

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# --- y_train, y_test are already defined (SPY Price) ---
# Fit ARIMA (no exogenous)
arima_model = SARIMAX(y_train, order=(1,1,1), seasonal_order=(0,0,0,0))
arima_res = arima_model.fit(disp=False)
print(arima_res.summary())

# Forecast for test horizon
n_forecast = len(y_test)
arima_forecast = arima_res.get_forecast(steps=n_forecast)
y_pred_arima = arima_forecast.predicted_mean
ci_arima = arima_forecast.conf_int()


Alternative Model 1 - ARIMAX Model with Exogenous Variables: Unemployment Rate (%), CPI, and CCI

In [ ]:
#  Fit ARIMAX Model
# ============================
# Target variable = SPY Price (you could also use % Change)
y_train = train["SPY Price"]
y_test = test["SPY Price"]

# Exogenous variables = macroeconomic monthly factors
exog_train = train[["Federal Interest Rate (%)", "Unemployment Rate (%)", "CPI", "CCI"]]
exog_test = test[["Federal Interest Rate (%)", "Unemployment Rate (%)", "CPI", "CCI"]]

# Fit ARIMAX (example order, tune later with AIC/BIC)
arimax_model = SARIMAX(y_train, exog=exog_train, order=(1,1,1), seasonal_order=(0,0,0,0))
arimax_res = arimax_model.fit(disp=False)
print(arimax_res.summary())

print("NaNs in train exog:\n", exog_train.isna().sum())
print("NaNs in test exog:\n", exog_test.isna().sum())

print("Any inf in train?", np.isinf(exog_train.values).any())
print("Any inf in test?", np.isinf(exog_test.values).any())

exog_train = exog_train.fillna(method="ffill").fillna(method="bfill")
exog_test = exog_test.fillna(method="ffill").fillna(method="bfill")

# assuming arimax_res (fitted SARIMAX with exog) exists and exog_test is ready
arimax_forecast = arimax_res.get_forecast(steps=n_forecast, exog=exog_test)
y_pred_arimax = arimax_forecast.predicted_mean
ci_arimax = arimax_forecast.conf_int()

# Point-forecast errors
rmse_arima  = np.sqrt(mean_squared_error(y_test, y_pred_arima))
mae_arima   = mean_absolute_error(y_test, y_pred_arima)

rmse_arimax = np.sqrt(mean_squared_error(y_test, y_pred_arimax))
mae_arimax  = mean_absolute_error(y_test, y_pred_arimax)

print("ARIMA  - RMSE: {:.3f}, MAE: {:.3f}, AIC: {:.3f}, BIC: {:.3f}".format(
    rmse_arima, mae_arima, arima_res.aic, arima_res.bic))
print("ARIMAX - RMSE: {:.3f}, MAE: {:.3f}, AIC: {:.3f}, BIC: {:.3f}".format(
    rmse_arimax, mae_arimax, arimax_res.aic, arimax_res.bic))

from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
import matplotlib.pyplot as plt

# ARIMA residuals
resid_arima = arima_res.resid.dropna()
# ARIMAX residuals
resid_arimax = arimax_res.resid.dropna()

# ACF plots
fig, axes = plt.subplots(2,1, figsize=(10,6))
plot_acf(resid_arima, lags=30, ax=axes[0], title="ARIMA Residuals ACF")
plot_acf(resid_arimax, lags=30, ax=axes[1], title="ARIMAX Residuals ACF")
plt.tight_layout()
plt.show()

# Ljung-Box (e.g., up to lag 10)
print("ARIMA Ljung-Box p-value (lag 10):", acorr_ljungbox(resid_arima, lags=[10], return_df=True)['lb_pvalue'].values[0])
print("ARIMAX Ljung-Box p-value (lag 10):", acorr_ljungbox(resid_arimax, lags=[10], return_df=True)['lb_pvalue'].values[0])

print(exog_test.isna().sum())
exog_test = exog_test.ffill()

# ============================
#  EGARCH on ARIMA residuals
# ============================

from arch import arch_model

# Use ARIMA residuals
residuals = arima_res.resid

# Fit EGARCH(1,1)
egarch = arch_model(residuals, vol="EGARCH", p=1, o=1, q=1, dist="t")
egarch_res = egarch.fit(disp="off")

print(egarch_res.summary())

# Plot conditional volatility
plt.figure(figsize=(12,6))
plt.plot(egarch_res.conditional_volatility, label="EGARCH Volatility (ARIMA residuals)")
plt.legend()
plt.show()

# -----------------------
# 6a. Forecast the ARIMA mean for the test period
# -----------------------
arima_forecast = arima_res.forecast(steps=len(y_test))
y_mean_pred = arima_forecast

# -----------------------
# 6b. Forecast EGARCH variance over the test horizon
# -----------------------
horizon = len(y_test)
egarch_forecast = egarch_res.forecast(horizon=horizon, method='simulation')

# Extract forecasted volatility (standard deviation)
vol_forecast = np.sqrt(egarch_forecast.variance.values[-1, :])

# Compute upper and lower bounds for 95% CI
upper = y_mean_pred + 1.96 * vol_forecast
lower = y_mean_pred - 1.96 * vol_forecast

# -----------------------
# 6c. Plot forecast with volatility
# -----------------------
plt.figure(figsize=(12,6))
plt.plot(y_train.index, y_train, label="Train")
plt.plot(y_test.index, y_test, label="Test", color="orange")
plt.plot(y_test.index, y_mean_pred, label="ARIMA Forecast", color="green")
plt.fill_between(y_test.index, lower, upper, color='green', alpha=0.2, label="95% CI")
plt.legend()
plt.title("SPY Price Forecast with ARIMA + EGARCH")
plt.show()

# -----------------------
# 6d. Evaluate forecast
# -----------------------
from sklearn.metrics import mean_squared_error, mean_absolute_error

rmse = np.sqrt(mean_squared_error(y_test, y_mean_pred))
mae = mean_absolute_error(y_test, y_mean_pred)
mape = np.mean(np.abs((y_test - y_mean_pred)/y_test)) * 100

print(f"Forecast Evaluation:\nRMSE: {rmse:.2f}\nMAE: {mae:.2f}\nMAPE: {mape:.2f}%")

# -----------------------
# Attach datetime index to forecast
# -----------------------
y_mean_pred_series = pd.Series(y_mean_pred, index=y_test.index)

# -----------------------
# Zoom-in window: last 60 days of train + test
# -----------------------
zoom_days = 60
zoom_start_date = y_train.index.max() - pd.Timedelta(days=zoom_days)

train_zoom = y_train[y_train.index >= zoom_start_date]
test_zoom = y_test[y_test.index >= zoom_start_date]
forecast_zoom = y_mean_pred_series[y_mean_pred_series.index >= zoom_start_date]

# Slice confidence intervals to match the zoomed test period
lower_zoom = lower[-len(test_zoom):]
upper_zoom = upper[-len(test_zoom):]

# -----------------------
# Plot zoomed-in forecast
# -----------------------
plt.figure(figsize=(12,6))
plt.plot(train_zoom.index, train_zoom, label="Train")
plt.plot(test_zoom.index, test_zoom, label="Test", color="orange")
plt.plot(forecast_zoom.index, forecast_zoom, label="ARIMA Forecast", color="green")
plt.fill_between(test_zoom.index, lower_zoom, upper_zoom, color='green', alpha=0.2, label="95% CI")
plt.legend()
plt.title("Zoomed-in 60-Day SPY Price Forecast (ARIMA + EGARCH)")
plt.show()

# -----------------------
# Attach datetime index to forecast
# -----------------------
y_mean_pred_series = pd.Series(y_mean_pred, index=y_test.index)

# -----------------------
# Zoom-in window: last 60 days of train + test
# -----------------------
zoom_days = 60
zoom_start_date = y_train.index.max() - pd.Timedelta(days=zoom_days)

train_zoom = y_train[y_train.index >= zoom_start_date]
test_zoom = y_test[y_test.index >= zoom_start_date]
forecast_zoom = y_mean_pred_series[y_mean_pred_series.index >= zoom_start_date]

# Slice confidence intervals to match the zoomed test period
lower_zoom = lower[-len(test_zoom):]
upper_zoom = upper[-len(test_zoom):]

# Combine train and test for actual price line
actual_zoom = pd.concat([train_zoom, test_zoom])

# -----------------------
# Plot zoomed-in forecast with actual price
# -----------------------
plt.figure(figsize=(12,6))
plt.plot(actual_zoom.index, actual_zoom, label="Actual Price", color="blue")
plt.plot(forecast_zoom.index, forecast_zoom, label="ARIMA Forecast", color="green")
plt.fill_between(test_zoom.index, lower_zoom, upper_zoom, color='green', alpha=0.2, label="95% CI")
plt.axvline(x=train_zoom.index.max(), color="red", linestyle="--", label="Train/Test Split")
plt.legend()
plt.title("Zoomed-in 60-Day SPY Price Forecast (ARIMA + EGARCH)")
plt.show()



Alternative Model 2 - ARIMAX with Exogenous Variable: Bitcoin Prices

In [ ]:
merged.head(20)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.arima.model import ARIMA

# ----------------------------
# Config (edit names if needed)
# ----------------------------
spy_col = 'SPY Price'
btc_col = 'Close'       # Bitcoin closing price
h = 44                 # future forecast horizon (about 2 months)

# ============================
# 1) Build the dataframe for use in the analysis
# ============================
if 'Date' in merged.columns:
    keep = ['Date', spy_col, btc_col]
    missing = [c for c in keep if c not in merged.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}. Available: {list(merged.columns)}")
    df_btc = merged[keep].copy()
    df_btc['Date'] = pd.to_datetime(df_btc['Date'], errors='coerce')
    df_btc = (df_btc.dropna(subset=['Date'])
                     .sort_values('Date')
                     .drop_duplicates('Date')
                     .set_index('Date'))
else:
    # Date already in the index
    for c in [spy_col, btc_col]:
        if c not in merged.columns:
            raise ValueError(f"Missing expected column: {c}. Available: {list(merged.columns)}")
    df_btc = merged[[spy_col, btc_col]].copy()
    df_btc.index = pd.to_datetime(df_btc.index, errors='coerce')
    df_btc = df_btc[~df_btc.index.isna()].sort_index()
    df_btc = df_btc[~df_btc.index.duplicated(keep='last')]

# Align to business days & clean numerics
df_btc = df_btc.asfreq('B')  # business-day grid
eps = 1e-8
for c in [spy_col, btc_col]:
    df_btc[c] = pd.to_numeric(df_btc[c], errors='coerce').clip(lower=eps).ffill().bfill()

# ============================
# 2) Endog/Exog
#    y = log(SPY), X = BTC log-returns
# ============================
log_spy = np.log(df_btc[spy_col])
btc_ret = np.log(df_btc[btc_col]).diff()

aligned = pd.concat([log_spy.rename('log_spy'),
                     btc_ret.rename('btc_ret')], axis=1).dropna()

y = aligned['log_spy']
X = aligned[['btc_ret']]

# ============================
# 3) Train/Test split
# ============================
split_point = (y.index.max() - pd.DateOffset(months=2)).floor('D')
y_train = y.loc[y.index <= split_point]
y_test  = y.loc[y.index >  split_point]
X_train = X.loc[X.index <= split_point]
X_test  = X.loc[X.index >  split_point]

print("Train:", y_train.index.min(), "to", y_train.index.max(), "|", len(y_train))
print("Test :", y_test.index.min(),  "to", y_test.index.max(),  "|", len(y_test))

# ============================
# 4) Fit ARIMAX on TRAIN
# ============================
order = (1, 1, 1)
res = ARIMA(endog=y_train, exog=X_train, order=order).fit()
print(res.summary())

# ============================
# 5) Forecast over TEST horizon
# ============================
n_forecast = len(y_test)
fc_test = res.get_forecast(steps=n_forecast, exog=X_test)

pred_log_test = pd.Series(fc_test.predicted_mean, index=y_test.index)
ci_log_test   = fc_test.conf_int()

# Back-transform to price for eval/plot
pred_test = np.exp(pred_log_test)
ci_lower_test = np.exp(ci_log_test.iloc[:, 0])
ci_upper_test = np.exp(ci_log_test.iloc[:, 1])
actual_test = np.exp(y_test)

mse  = mean_squared_error(actual_test, pred_test)
rmse = float(np.sqrt(mse))
mae  = mean_absolute_error(actual_test, pred_test)
print(f"Test RMSE: {rmse:,.4f} | MAE: {mae:,.4f}")

# ============================
# 6) Refit on FULL data, then 100-BD future forecast
# ============================
res_full = ARIMA(endog=y, exog=X, order=order).fit()

future_idx = pd.bdate_range(df_btc.index.max() + pd.offsets.BusinessDay(), periods=h)

last_btc_ret = float(X.iloc[-1, 0])
X_future = pd.DataFrame({'btc_ret': last_btc_ret}, index=future_idx)

fc_future = res_full.get_forecast(steps=h, exog=X_future)
pred_log_future = pd.Series(fc_future.predicted_mean, index=future_idx)
ci_log_future   = fc_future.conf_int()

price_future     = np.exp(pred_log_future)
ci_lower_future  = np.exp(ci_log_future.iloc[:, 0])
ci_upper_future  = np.exp(ci_log_future.iloc[:, 1])

# Save table
forecast_table = pd.DataFrame({
    'Date': price_future.index,
    'Forecast Price': price_future.round(2),
    'Lower 95% CI': ci_lower_future.round(2),
    'Upper 95% CI': ci_upper_future.round(2)
}).reset_index(drop=True)

print(forecast_table.head(10))
forecast_table.to_csv('spy_100bd_future_forecast.csv', index=False)
print("Saved 100-business-day forecast to spy_44bd_future_forecast.csv")

# ============================
# 7) Plot: history, test forecast, future forecast
# ============================
plt.figure(figsize=(14, 6))
plt.plot(df_btc.index, df_btc[spy_col], label='Observed SPY', color='black', linewidth=1)
plt.axvline(split_point, ls='--', alpha=0.7, label='Train/Test split')

# Test forecast
plt.plot(pred_test.index, pred_test, label='ARIMAX forecast (test)', linewidth=2)
plt.fill_between(pred_test.index, ci_lower_test, ci_upper_test, alpha=0.25, label='95% CI (test)')

# Future forecast
plt.plot(price_future.index, price_future, label='Future forecast (44 BD)', linewidth=2)
plt.fill_between(price_future.index, ci_lower_future, ci_upper_future, alpha=0.25, label='95% CI (future)')

plt.title('SPY — ARIMAX with BTC returns: test performance + 44-BD future forecast')
plt.xlabel('Date')
plt.ylabel('SPY Price')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


Alternative Model 3 - ARIMAX with Exogenous Variables: VIX and UST10Y

In [ ]:
# === SPY Forecast (Oct 1 → Dec 1, 2025) Script ===
# Time Series Project Dataset-1.csv with columns: Date, SPY Price, VIX Price
#===================

# Load SPY & VIX
# ---------------------------
from google.colab import drive
drive.mount('/content/drive')
df_VIX = pd.read_excel('/content/drive/MyDrive/Time Series Project/Time Series Project Dataset.xlsx', sheet_name='Daily Data',
                 usecols=['Date', 'SPY Price', 'VIX Price'])

df_VIX['Date'] = pd.to_datetime(df_VIX['Date'], errors='coerce')
df_VIX = (df_VIX.dropna(subset=['Date'])
        .sort_values('Date')
        .drop_duplicates('Date')
        .set_index('Date'))

dp = pd.DataFrame({
    'price': pd.to_numeric(df_VIX['SPY Price'], errors='coerce'),
    'VIX'  : pd.to_numeric(df_VIX['VIX Price'], errors='coerce'),
}).dropna()

# Training cutoff & forecast horizon
# ---------------------------
target_cutoff = pd.Timestamp('2025-09-30')
train_end = min(target_cutoff, dp.index.max())
train = dp.loc[:train_end].copy()

start, end = pd.Timestamp('2025-10-01'), pd.Timestamp('2025-12-01')
def trading_index(start, end):
    try:
        import pandas_market_calendars as mcal
        nyse = mcal.get_calendar('NYSE')
        sched = nyse.schedule(start_date=start, end_date=end)
        out = mcal.date_range(sched, frequency='1D')
        return pd.DatetimeIndex(out.tz_localize(None))
    except Exception:
        return pd.bdate_range(start=start, end=end)

forecast_index = trading_index(start, end)

print(f"Training range: {train.index.min().date()} → {train.index.max().date()}  (n={len(train)})")
print(f"Forecast horizon: {forecast_index[0].date()} → {forecast_index[-1].date()} (N={len(forecast_index)})")

# Fetch 10Y (DGS10) from FRED for TRAINING ONLY (percent levels)
# ---------------------------
idx_train = train.index
use_exog, ust = False, None
try:
    from pandas_datareader import data as web
    ust = web.DataReader('DGS10', 'fred', idx_train.min(), idx_train.max()).rename(columns={'DGS10':'UST10Y'})
    ust = ust.reindex(idx_train).ffill()
    use_exog = not ust.empty
except Exception:
    use_exog = False

print(f"FRED DGS10 available: {use_exog}")

# Build TRAINING exog (only if DGS10 available)
#     - VIX_ret_l1  = lag1 log-return of VIX
#     - UST10Y_dbp_l4 = lag4 daily change of 10Y in basis points
# ---------------------------
if use_exog:
    vix_aligned = train['VIX'].reindex(idx_train).ffill()
    vix_ret = np.log(vix_aligned).diff()                      # stationary
    ust_dbp = ust['UST10Y'].diff().mul(100)                   # percent → bp change
    X_train = pd.DataFrame({
        'VIX_ret_l1': vix_ret.shift(1),
        'UST10Y_dbp_l4': ust_dbp.shift(4),
    }, index=idx_train).dropna()
    y_train = train['price'].reindex(X_train.index)
else:
    X_train = None
    y_train = train['price']

# Fit models on identical rows
# ---------------------------
m_arima = SARIMAX(y_train, order=(0,1,0), trend='t',
                  enforce_stationarity=False, enforce_invertibility=False
                 ).fit(method='lbfgs', maxiter=200, disp=False)

if use_exog:
    m_arimax = SARIMAX(y_train, order=(0,1,0), trend='t', exog=X_train,
                       enforce_stationarity=False, enforce_invertibility=False
                      ).fit(method='lbfgs', maxiter=200, disp=False)
    print("\nAIC — ARIMA:", round(m_arima.aic,2), "| ARIMAX:", round(m_arimax.aic,2))
else:
    print("\nAIC — ARIMA:", round(m_arima.aic,2), "| ARIMAX: skipped (no DGS10)")

# Overlay inside training (last-fold path) — ARIMA vs ARIMAX
# ---------------------------
H = 44  # ~2 months
idx = y_train.index
last_origin = idx[-H-1] if len(idx) > H+1 else idx[-2]

y_end = y_train.loc[:last_origin]
r_arima = SARIMAX(y_end, order=(0,1,0), trend='t',
                  enforce_stationarity=False, enforce_invertibility=False
                 ).fit(method='lbfgs', maxiter=200, disp=False)

start_pos = idx.get_loc(last_origin) + 1
end_pos   = min(start_pos + H - 1, len(idx) - 1)
h_steps   = end_pos - start_pos + 1
fc_a = r_arima.get_forecast(steps=h_steps)
path_a = pd.Series(fc_a.predicted_mean.values, index=idx[start_pos:start_pos+h_steps])

plt.figure(figsize=(12,5))
hist_start = y_train.index.max() - pd.DateOffset(months=12)
plt.plot(dp.loc[hist_start:y_train.index.max(), 'price'], label='History (last 12m)')
plt.plot(path_a.index, path_a.values, label='ARIMA last-fold path')
if use_exog:
    X_end = X_train.loc[:last_origin]
    r_arimax = SARIMAX(y_end, order=(0,1,0), trend='t', exog=X_end,
                       enforce_stationarity=False, enforce_invertibility=False
                      ).fit(method='lbfgs', maxiter=200, disp=False)
    fc_x = r_arimax.get_forecast(steps=h_steps, exog=X_train.iloc[start_pos:start_pos+h_steps])
    path_x = pd.Series(fc_x.predicted_mean.values, index=idx[start_pos:start_pos+h_steps])
    plt.plot(path_x.index, path_x.values, label='ARIMAX (VIX l1 + UST l4) last-fold path')
plt.title("Overlay: ARIMA vs ARIMAX (inside training, last fold)")
plt.xlabel('Date'); plt.ylabel('Price ($)'); plt.grid(True); plt.legend()
plt.tight_layout(); plt.show()

# Build FORECAST exog for Oct→Dec — default to "no shocks"
#     (If desired later, you can pass future VIX/UST10Y levels.)
# ---------------------------
def build_X_fore(forecast_index, vix_future=None, ust10_future=None):
    Xf = pd.DataFrame({'VIX_ret_l1': 0.0, 'UST10Y_dbp_l4': 0.0}, index=forecast_index)
    if (vix_future is not None) and ('VIX' in train.columns):
        vf = vix_future.copy()
        if isinstance(vf, pd.Series): vf = vf.to_frame('VIX')
        # stitch last training VIX level
        last_vix = train[['VIX']].iloc[[-1]]
        all_vix = pd.concat([last_vix, vf[['VIX']]])
        all_vix = all_vix.reindex(pd.DatetimeIndex([last_vix.index[-1]]).append(forecast_index))
        vix_ret_f = np.log(all_vix['VIX']).diff().shift(1)
        Xf['VIX_ret_l1'] = vix_ret_f.reindex(forecast_index).fillna(0.0)
    if (ust10_future is not None) and (use_exog):
        uf = ust10_future.copy()
        if isinstance(uf, pd.Series): uf = uf.to_frame('UST10Y')
        last_ust = ust[['UST10Y']].iloc[[-1]]
        all_ust = pd.concat([last_ust, uf[['UST10Y']]])
        all_ust = all_ust.reindex(pd.DatetimeIndex([last_ust.index[-1]]).append(forecast_index))
        ust_dbp_f = all_ust['UST10Y'].diff().mul(100).shift(4)
        Xf['UST10Y_dbp_l4'] = ust_dbp_f.reindex(forecast_index).fillna(0.0)
    return Xf

X_fore = None
if use_exog:
    # default no-shock exog (zeros); plug future levels to override
    X_fore = build_X_fore(forecast_index, vix_future=None, ust10_future=None)

# Final FORECAST (Oct→Dec 2025) + plot & CSV
# ---------------------------
if use_exog:
    final_model = SARIMAX(y_train, order=(0,1,0), trend='t', exog=X_train,
                          enforce_stationarity=False, enforce_invertibility=False
                         ).fit(method='lbfgs', maxiter=200, disp=False)
    fc = final_model.get_forecast(steps=len(forecast_index), exog=X_fore)
else:
    final_model = SARIMAX(y_train, order=(0,1,0), trend='t',
                          enforce_stationarity=False, enforce_invertibility=False
                         ).fit(method='lbfgs', maxiter=200, disp=False)
    fc = final_model.get_forecast(steps=len(forecast_index))

pred = fc.predicted_mean
ci80 = fc.conf_int(alpha=0.20)
ci95 = fc.conf_int(alpha=0.05)

forecast_df = pd.DataFrame({
    'Point'      : pred.to_numpy(),
    'PI80_Lower' : ci80.iloc[:,0].to_numpy(),
    'PI80_Upper' : ci80.iloc[:,1].to_numpy(),
    'PI95_Lower' : ci95.iloc[:,0].to_numpy(),
    'PI95_Upper' : ci95.iloc[:,1].to_numpy(),
}, index=forecast_index)

# Baseline ARIMA overlay line (trained on y_train only)
base_model = SARIMAX(y_train, order=(0,1,0), trend='t',
                     enforce_stationarity=False, enforce_invertibility=False
                    ).fit(method='lbfgs', maxiter=200, disp=False)
base_fc = base_model.get_forecast(steps=len(forecast_index))
base_df = pd.DataFrame({'Point': base_fc.predicted_mean.to_numpy()}, index=forecast_index)

print("\nForecast preview:")
print(forecast_df.head())

plt.figure(figsize=(12,5))
hist2 = dp.loc[(y_train.index.max() - pd.DateOffset(months=12)):y_train.index.max(), 'price']
plt.plot(hist2.index, hist2.values, label='History (last 12m)')
plt.plot(base_df.index, base_df['Point'], label='ARIMA forecast')
plt.plot(forecast_df.index, forecast_df['Point'],
         label=('ARIMAX forecast' if use_exog else 'ARIMA (final)'))
plt.fill_between(forecast_df.index, forecast_df['PI95_Lower'], forecast_df['PI95_Upper'],
                 alpha=0.15, label='95% PI')
plt.fill_between(forecast_df.index, forecast_df['PI80_Lower'], forecast_df['PI80_Upper'],
                 alpha=0.25, label='80% PI')
plt.axvline(y_train.index.max(), linestyle='--', linewidth=1, color='k')
plt.title("SPY — Final Forecast (Oct 1 → Dec 1, 2025)")
plt.xlabel('Date'); plt.ylabel('Price ($)'); plt.grid(True); plt.legend()
plt.tight_layout(); plt.show()

outname = 'spy_forecast_oct_dec_2025_arimax.csv' if use_exog else 'spy_forecast_oct_dec_2025_arima.csv'
forecast_df.to_csv(outname)
print(f"\nSaved: {outname}")

# Anchor dates for report (if present in horizon)
for d in [pd.Timestamp('2025-10-01'), pd.Timestamp('2025-11-15'), pd.Timestamp('2025-12-01')]:
    if d in forecast_df.index:
        r = forecast_df.loc[d]
        print(f"{d.date()}  Point={r['Point']:.2f}  95%PI[{r['PI95_Lower']:.2f}, {r['PI95_Upper']:.2f}]")

Alternative Model 4 - ARIMAX with Exogenous Variables: VIX and Federal Interest Rate

In [ ]:
!pip install pmdarima

In [ ]:
!pip uninstall -y numpy pandas
!pip install numpy==1.26.4 pandas==2.2.2

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter("ignore", ConvergenceWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from arch import arch_model
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import t
import pandas_datareader.data as web
from sklearn.metrics import mean_squared_error, mean_absolute_error
from google.colab import drive
from statsmodels.tsa.stattools import grangercausalitytests
from pmdarima import auto_arima
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

drive.mount('/content/drive')

spy = pd.read_excel('/content/drive/MyDrive/Time Series Project/Time Series Project Dataset.xlsx', sheet_name='Daily Data')
monthly = pd.read_excel('/content/drive/MyDrive/Time Series Project/Time Series Project Dataset.xlsx', sheet_name='Monthly Data')
btc = pd.read_csv('/content/drive/MyDrive/Time Series Project/Bitcoin_history_data.csv')

spy['Month'] = spy['Date'].dt.to_period('M')
monthly['Month'] = monthly['Month'].dt.to_period('M')
merged_df = pd.merge(spy, monthly, on='Month', how='left')
merged_df['Date'] = pd.to_datetime(merged_df['Date']) # Convert 'Date' column to datetime
btc['Date'] = pd.to_datetime(btc['Date'])
merged_df = pd.merge(merged_df, btc, on='Date', how='left')
display(merged_df.head())

# Train and test split
last_date = merged_df['Date'].max()
validation_start = last_date - pd.DateOffset(months=2)

merged_df['VIX_log'] = np.log(merged_df['VIX Price']) # Transform VIX
merged_df['BTC_log'] = np.log(merged_df['Close'])
spySARIMAX = merged_df.dropna(subset=['% Change', 'VIX_log', 'BTC_log', 'Federal Interest Rate (%)'])
spySARIMAXtrain = spySARIMAX[spySARIMAX['Date'] < validation_start].copy()
spySARIMAXvalidation = spySARIMAX[spySARIMAX['Date'] >= validation_start].copy()

# ---- ACF/PACF Plots ----
fig, axes = plt.subplots(1, 2, figsize=(12,4))
plot_acf(spySARIMAX['% Change'], lags=30, ax=axes[0])
plot_pacf(spySARIMAX['% Change'], lags=30, ax=axes[1])
plt.show()

exog = ['VIX_log', 'BTC_log', 'Federal Interest Rate (%)']

# Perform Granger Causality tests
print('VIX')
grangercausalitytests(spySARIMAX[['% Change', 'VIX_log']].dropna(), maxlag=5)

print('Interest Rates')
grangercausalitytests(spySARIMAX[['% Change', 'Federal Interest Rate (%)']].dropna(), maxlag=5)

stepwise = auto_arima(
    spySARIMAXtrain['% Change'],
    exogenous=spySARIMAXtrain[exog],
    seasonal=False,
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    max_p=5, max_q=5
)
print(stepwise.summary())

# Forecast same length as validation set
n_periods = len(spySARIMAXvalidation)

forecast, conf_int = stepwise.predict(n_periods=n_periods, return_conf_int=True)
forecast = pd.Series(forecast, index=spySARIMAXvalidation.index)

plt.figure(figsize=(12,6))
plt.plot(spySARIMAXtrain['Date'], spySARIMAXtrain['% Change'], label="Training Data", alpha=0.7)
plt.plot(spySARIMAXvalidation['Date'], spySARIMAXvalidation['% Change'], label="Actual Validation Data", color="blue", alpha=0.7)
plt.plot(spySARIMAXvalidation['Date'], forecast, label="Predicted (Validation)", color="orange")
plt.title("SPY Returns Forecast (Train vs Validation)")
plt.xlabel("Date")
plt.ylabel("Daily % Change")
plt.legend()
plt.fill_between(
    spySARIMAXvalidation['Date'],
    conf_int[:, 0],  # lower bounds
    conf_int[:, 1],  # upper bounds
    color="orange",
    alpha=0.2,
    label="Confidence Interval"
)

plt.show()

plt.figure(figsize=(12,6))
plt.plot(spySARIMAXvalidation['Date'], spySARIMAXvalidation['% Change'],
         label="Actual Validation Data", color="blue", alpha=0.7)
plt.plot(spySARIMAXvalidation['Date'], forecast,
         label="Predicted (Validation)", color="orange")

plt.title("SPY Returns Forecast (Validation)")
plt.xlabel("Date")
plt.ylabel("Daily % Change")
plt.legend()

# Zoom the x-axis to validation period
plt.xlim(spySARIMAXvalidation['Date'].min(), spySARIMAXvalidation['Date'].max())

plt.show()

last_train_price = spySARIMAXtrain['SPY Price'].iloc[-1]

# Forecasted % changes are in "forecast" (validation index aligned)
price_forecast = last_train_price * (1 + forecast).cumprod()

# Convert CI bounds into prices (apply same cumulative transform)
price_conf_int_lower = last_train_price * (1 + conf_int[:, 0]).cumprod()
price_conf_int_upper = last_train_price * (1 + conf_int[:, 1]).cumprod()

# Actual validation prices
actual_prices = spySARIMAXvalidation['SPY Price']

plt.figure(figsize=(12,6))
plt.plot(spySARIMAXtrain['Date'], spySARIMAXtrain['SPY Price'], label="Training Prices", alpha=0.7)
plt.plot(spySARIMAXvalidation['Date'], actual_prices, label="Actual Validation Prices", color="blue", alpha=0.7)
plt.plot(spySARIMAXvalidation['Date'], price_forecast, label="Forecasted Prices", color="orange")
plt.title("SPY Price Forecast")
plt.xlabel("Date")
plt.ylabel("SPY Price")
plt.legend()
plt.fill_between(
    spySARIMAXvalidation['Date'],
    price_conf_int_lower,
    price_conf_int_upper,
    color="orange",
    alpha=0.2,
    label="Confidence Interval"
)

plt.show()

plt.figure(figsize=(12,6))

# Plot only validation + forecast period
plt.plot(spySARIMAXvalidation['Date'], actual_prices, label="Actual Validation Prices", color="blue", alpha=0.7)
plt.plot(spySARIMAXvalidation['Date'], price_forecast, label="Forecasted Prices", color="orange")

plt.title("SPY Price Forecast (Validation)")
plt.xlabel("Date")
plt.ylabel("SPY Price")
plt.legend()

# Zoom y-axis to focus around actual SPY levels
plt.ylim(
    actual_prices.min() * 0.98,
    actual_prices.max() * 1.02
)

plt.show()

summary = stepwise.summary()

# Extract coefficients table
coefs = summary.tables[1]
coef_df = pd.DataFrame(coefs.data[1:], columns=coefs.data[0])
coef_df.columns = ["Variable", "Coef", "Std Err", "z", "P>|z|", "[0.025", "0.975]"]

for col in ["Coef", "Std Err", "z", "P>|z|", "[0.025", "0.975]"]:
    coef_df[col] = pd.to_numeric(coef_df[col], errors="coerce")

coef_df["Significant"] = coef_df["P>|z|"] < 0.05

metrics = {
    "Model": str(stepwise.order),
    "Log Likelihood": stepwise.arima_res_.llf,
    "AIC": stepwise.aic(),
    "BIC": stepwise.bic(),
    "HQIC": stepwise.arima_res_.hqic
}

print("=== ARIMA Model Summary ===")
for k,v in metrics.items():
    print(f"{k:>15}: {v}")

print("\n=== Coefficients (significant ones marked) ===")
print(coef_df.to_string(index=False))

In [ ]:
# Fit GARCH(1,1) with exogenous variable in variance equation
model = arch_model(
    spySARIMAXtrain["% Change"], # Pass only the target variable here
    vol="GARCH",
    p=1, q=1,
    mean="Constant",
    x=spySARIMAX[exog],
    dist='normal'
)

# Add exogenous variable to variance equation
res = model.fit(disp="off")

In [ ]:
from arch.univariate import arch_model

am = arch_model(
    spySARIMAXtrain["% Change"],
    #mean='ARX',  # ARX allows exogenous variables in mean
    lags=1,
    vol='GARCH',
    p=1, q=1,
    dist='normal',
    #x=spySARIMAXtrain[exog]       # exogenous variables for mean
)

res = am.fit()
print(res.summary())

In [ ]:
print(res.summary())

In [ ]:
mu = res.params['mu']  # constant mean
fitted_vals = mu + res.resid  # in-sample returns = mean + residuals
last_fitted_values = fitted_vals.tail(len(spySARIMAXvalidation))

plt.figure(figsize=(12,6))
plt.plot(spySARIMAXtrain['Date'], spySARIMAXtrain['% Change'],
         label="Training Data")
plt.plot(spySARIMAXvalidation['Date'], spySARIMAXvalidation['% Change'],
         label="Training Data", color = 'blue', alpha=0.6)
plt.plot(spySARIMAXvalidation['Date'], last_fitted_values,
         color="orange", label="GARCH In-Sample Fit")

plt.title("SPY GARCH In-Sample Fit")
plt.xlabel("Date")
plt.ylabel("Daily % Change")
plt.legend()
plt.show()
